### Schwierigkeiten – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024), 1. Runde, Aufgabe 2*

Dua L. Graph ist Lehrerin für Informatik. Im Laufe der Jahre hat sie eine Reihe von Klausuraufgaben ausgearbeitet. Damit hat sie sich
so viel Mühe gemacht, dass sie ihre Aufgaben – mit zeitlichem Abstand und nur leicht verändert – immer wieder verwendet.

Dua leitet aus den Ergebnissen früherer Klausuren ab, welche Aufgaben darin leichter waren als andere. Zum Beispiel hat sie aus vier
alten Klausuren die folgenden Schwierigkeitsabstufungen abgeleitet (`<` bedeutet „ist leichter als“):

| Klausur | Schwierigkeitsabstufung |
|:-:|---|
| 1 | B < A < D < F *(und damit auch B < D, B < F und A < F)* |
| 2 | D < F < G |
| 3 | A < E < D < C |
| 4 | G < F < C |

Wenn Dua aus bisherigen Aufgaben eine neue Klausur zusammen stellt, möchte sie die Aufgaben nach Schwierigkeit anordnen, leichte
Aufgaben zuerst. In der nächsten Klausur möchte Dua die Aufgaben B, C, D, E und F verwenden. Sie ordnet die Aufgaben so an: B; E; D; F; C.
Geschafft! Ihre Anordnung ist eine gute Anordnung, denn sie steht zu keiner vorherigen Schwierigkeitsabstufung im Widerspruch.

Dua ist erleichtert. Als begeisterte Informatikerin will sie sich für die Zukunft das Leben einfacher machen und die Anordnung ihrer
Aufgaben automatisieren. Sie stellt aber fest, dass das nicht immer leicht ist: Wenn sie zum Beispiel die Aufgaben F und G auswählt, ist
laut Klausur 2 Aufgabe F einfacher als G, aber laut Klausur 4 Aufgabe G einfacher als F. Eine solche Situation nennt Dua einen Konflikt.

#### Aufgabe 2

Schreibe ein Programm, das Dua hilft. Es soll die Schwierigkeitsabstufungen einer Reihe alter Klausuren sowie eine Menge von Aufgaben aus
diesen Klausuren einlesen und eine gute Anordnung dieser Aufgaben ausgeben.

1. Gehe zunächst davon aus, dass es in den alten Klausuren keine Konflikte gibt.
2. Überlege, wie dein Programm mit Konflikten umgehen könnte, und erweitere es entsprechend.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `schwierigkeiten0.txt` bis `schwierigkeiten5.txt`. Jede Datei ist so aufgebaut:

- 1. Zeile: drei Zahlen `n m k`, die Zahl der alten Klausuren, die Gesamtzahl der Aufgaben und die Zahl der Aufgaben, die angeordnet werden sollen.
- Dann `n` Zeilen mit den Abstufungen der alten Klausuren in der Form `B < A < D < F`.
- Die Aufgaben heißen `A`, `B`, … (die ersten `m` Buchstaben des Alphabets).
- Die letzte Zeile enthält die `k` Aufgaben, für die eine gute Anordnung gesucht wird.

```
4 7 5
B < A < D < F
D < F < G
A < E < D < C
G < F < C
B C D E F
```

`schwierigkeiten0.txt` ist das Beispiel aus der Aufgabenstellung.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graph:** Jede Aufgabe ist ein **Knoten**. Für „X ist leichter als Y“ zeichnet man einen **Pfeil** von X nach Y. In `B < A < D < F` genügen die
  Pfeile zwischen **direkt aufeinanderfolgenden** Aufgaben, denn `B < D` ergibt sich aus dem Pfad `B → A → D`.
- **Pfade und Erreichbarkeit:** X ist leichter als Y, wenn es einen **Pfad** von X nach Y gibt. Das gilt auch über mehrere Klausuren hinweg.
  Ob ein Pfad existiert, findet man mit einer Suche, die jeden Knoten nur einmal besucht (*Tiefensuche*).
- **Topologische Sortierung:** Eine Anordnung der Knoten, in der alle Pfeile von vorn nach hinten zeigen. Man findet sie, indem man immer
  eine Aufgabe nimmt, die keine noch nicht eingeordnete leichtere Aufgabe mehr hat (*Algorithmus von Kahn*).
- **Konflikt = Kreis:** Gibt es einen Pfad von X nach Y **und** von Y nach X, geht keine Anordnung. Das erkennt man daran, dass die
  Sortierung nicht zu Ende kommt.
- **Gleich schwer:** Aufgaben, die sich gegenseitig erreichen, behandelt man als **gleich schwer**. Das sind die *starken
  Zusammenhangskomponenten* des Graphen. Zwischen den Gruppen gibt es keine Kreise mehr, man kann also die **Gruppen** topologisch sortieren.
- **Alle Aufgaben sortieren:** Man sortiert zuerst **alle** Aufgaben und lässt dann die nicht benötigten weg. So bleibt die Beziehung
  „über eine nicht ausgewählte Aufgabe“ erhalten.

**Nützliche Funktionen und Bibliotheken**
- `set` (Menge) und `dict` (Zuordnung Aufgabe → Menge), Listen als Stapel (`append`, `pop`)
- `chr(65 + i)` macht aus 0, 1, 2, … die Buchstaben A, B, C, …

## Die Idee

1. Aus den Klausuren einen **Graphen** bauen: ein Pfeil von jeder Aufgabe zur nächstschwierigeren Aufgabe derselben Klausur.
2. **Ohne Konflikte** die Aufgaben topologisch sortieren, aus der Sortierung die gewünschten Aufgaben herausnehmen, fertig.
3. **Mit Konflikten** (Kreise im Graphen) die Aufgaben in Gruppen gleich schwerer Aufgaben einteilen: Wer sich gegenseitig erreicht, kommt in
   eine Gruppe. Dann die **Gruppen** topologisch sortieren und innerhalb einer Gruppe in beliebiger Reihenfolge ausgeben.
4. Die Anordnung **nachprüfen**: Alle Aufgaben da, und jede Aufgabe, die eindeutig leichter ist als eine andere, steht vor ihr.

## Schritt 1: Die Eingabe lesen

Die erste Zeile enthält `n m k`. Die folgenden `n` Zeilen sind die Klausuren, zum Beispiel `B < A < D < F`. Wir teilen jede Zeile
bei den `<`-Zeichen (`split('<')`) und entfernen die Leerzeichen (`strip()`). Die letzte Zeile enthält die gewünschten Aufgaben.
Die Aufgaben heißen `A`, `B`, … Mit `chr(65 + i)` bekommt man die Buchstaben (`chr(65)` ist `'A'`).

In [ ]:
def lies_aufgabe(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    n, m, k = zeilen[0].split()
    n = int(n)
    m = int(m)
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    aufgaben = []
    for i in range(m):
        aufgaben.append(chr(65 + i))
    return aufgaben, klausuren, gewaehlt


aufgaben, klausuren, gewaehlt = lies_aufgabe('beispieldaten/schwierigkeiten0.txt')
print('Aufgaben:', aufgaben)
print('Klausuren:', klausuren)
print('Gewünscht:', gewaehlt)

Aufgaben: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Klausuren: [['B', 'A', 'D', 'F'], ['D', 'F', 'G'], ['A', 'E', 'D', 'C'], ['G', 'F', 'C']]
Gewünscht: ['B', 'C', 'D', 'E', 'F']


## Schritt 2: Der Graph

Wir speichern für jede Aufgabe die Menge der **direkt schwierigeren** Aufgaben in einem `dict` `schwerer`. Aus jeder Klausur nehmen wir die
Paare aufeinanderfolgender Aufgaben. Eine **Menge** speichert eine Beziehung nur einmal, auch wenn sie in mehreren Klausuren vorkommt.

In [ ]:
def baue_graph(aufgaben, klausuren):
    schwerer = {}
    for a in aufgaben:
        schwerer[a] = set()
    for klausur in klausuren:
        for i in range(len(klausur) - 1):
            schwerer[klausur[i]].add(klausur[i + 1])
    return schwerer


schwerer = baue_graph(aufgaben, klausuren)
for a in aufgaben:
    print(a, 'ist direkt leichter als', sorted(schwerer[a]))

A ist direkt leichter als ['D', 'E']
B ist direkt leichter als ['A']
C ist direkt leichter als []
D ist direkt leichter als ['C', 'F']
E ist direkt leichter als ['D']
F ist direkt leichter als ['C', 'G']
G ist direkt leichter als ['F']


Zum Beispiel ist `D` direkt leichter als `F` (Klausuren 1 und 2) und `C` (Klausur 3). `G` ist direkt leichter als `F` und `F` direkt leichter als
`G`: Das ist der Konflikt aus der Aufgabe, ein Kreis `F → G → F`.

## Schritt 3: Ohne Konflikte sortieren (Kahn)

Wenn es keinen Konflikt gibt, sortieren wir topologisch. Wir halten für jede Aufgabe fest, **wie viele Aufgaben direkt leichter** sind und noch nicht
eingeordnet wurden (`wartet_auf`). Aufgaben mit `0` können wir sofort ausgeben. Nach dem Ausgeben einer Aufgabe verringern wir den Zähler
bei allen direkt schwierigeren Aufgaben, und wer dabei auf `0` kommt, wird als Nächstes bereit.

Kommt jede Aufgabe an die Reihe, ist die Sortierung vollständig. Bleiben Aufgaben übrig, hängen sie in einem **Kreis**, also in einem
Konflikt.

In [ ]:
def sortiere(aufgaben, schwerer):
    """Topologische Sortierung. Ergebnis: die Aufgaben in einer erlaubten Reihenfolge (bei Kreisen fehlen die Aufgaben im Kreis)."""
    wartet_auf = {}
    for a in aufgaben:
        wartet_auf[a] = 0
    for a in aufgaben:
        for b in schwerer[a]:
            wartet_auf[b] = wartet_auf[b] + 1
    bereit = []
    for a in aufgaben:
        if wartet_auf[a] == 0:
            bereit.append(a)
    reihenfolge = []
    while len(bereit) > 0:
        a = bereit.pop(0)                                  # die vorderste bereite Aufgabe
        reihenfolge.append(a)
        for b in sorted(schwerer[a]):
            wartet_auf[b] = wartet_auf[b] - 1
            if wartet_auf[b] == 0:
                bereit.append(b)
    return reihenfolge


aufgaben1, klausuren1, gewaehlt1 = lies_aufgabe('beispieldaten/schwierigkeiten1.txt')
schwerer1 = baue_graph(aufgaben1, klausuren1)
reihenfolge = sortiere(aufgaben1, schwerer1)
print('schwierigkeiten1: alle Aufgaben sortiert:', reihenfolge)
anordnung = []
for a in reihenfolge:
    if a in gewaehlt1:
        anordnung.append(a)
print('gewünscht', gewaehlt1, '-> Anordnung', anordnung)
print()
print('schwierigkeiten0 mit Konflikt:', sortiere(aufgaben, schwerer), 'statt aller', len(aufgaben), 'Aufgaben')

schwierigkeiten1: alle Aufgaben sortiert: ['A', 'B', 'E', 'C', 'G', 'F', 'D']
gewünscht ['A', 'C', 'D', 'F', 'G'] -> Anordnung ['A', 'C', 'G', 'F', 'D']

schwierigkeiten0 mit Konflikt: ['B', 'A', 'E', 'D'] statt aller 7 Aufgaben


Bei `schwierigkeiten1.txt` gibt es keinen Konflikt. Die Sortierung enthält alle sieben Aufgaben, und die gewünschten fünf (`A C D F G`) stehen in der
richtigen Reihenfolge. Nur die Aufgaben, die nicht gewünscht sind (hier `B` und `E`), lassen wir weg. Sie haben aber vorher mitgewirkt: Zum Beispiel
ergibt `A < E < C` und `E < G < F`, dass `A` vor `G` stehen muss, obwohl `E` nicht gewünscht ist.

Bei `schwierigkeiten0.txt` (Konflikt zwischen `F` und `G`) kommt die Sortierung nicht zu Ende: Es fehlen die Aufgaben im Kreis und alles, was von ihnen
abhängt. Dafür brauchen wir etwas Neues.

## Schritt 4: Wer ist leichter als wer? Erreichbarkeit

Aufgabe `X` ist leichter als `Y`, wenn es im Graphen einen **Pfad** von `X` nach `Y` gibt, nicht nur einen direkten Pfeil. `erreichbar(schwerer, a)` findet alle
Aufgaben, die von `a` aus über Pfeile erreichbar sind, also alle Aufgaben, die **schwerer** sind als `a`.

Das ist eine **Tiefensuche**. Wir haben einen Stapel `offen` mit Aufgaben, deren Nachfolger wir noch anschauen müssen, und eine Menge `gesehen`.
Wir nehmen eine Aufgabe vom Stapel, und jeden direkt schwierigeren Nachbarn, den wir noch nicht kennen, legen wir in `gesehen` und auf den Stapel.

In [ ]:
def erreichbar(schwerer, a):
    """Alle Aufgaben, die (über beliebig viele Pfeile) schwerer sind als a."""
    gesehen = set()
    offen = [a]
    while len(offen) > 0:
        x = offen.pop()
        for y in schwerer[x]:
            if y not in gesehen:
                gesehen.add(y)
                offen.append(y)
    return gesehen


print('schwerer als B:', sorted(erreichbar(schwerer, 'B')))
print('schwerer als F:', sorted(erreichbar(schwerer, 'F')))
print('schwerer als G:', sorted(erreichbar(schwerer, 'G')))

schwerer als B: ['A', 'C', 'D', 'E', 'F', 'G']
schwerer als F: ['C', 'F', 'G']
schwerer als G: ['C', 'F', 'G']


Alle anderen Aufgaben sind schwerer als `B`, denn `B` ist ganz vorn: `B < A`, und von `A` aus geht es weiter zu `D`, `E`, `F`, `G` und `C`. Von `F` aus erreicht man
`G` und `C`, und **auch `F` selbst**, denn `F → G → F` ist ein Kreis. Genauso erreicht `G` das `F`. Das ist der Konflikt: `F` und `G` erreichen sich gegenseitig.

## Schritt 5: Konflikte, gleich schwere Aufgaben

Wie soll das Programm mit einem Konflikt umgehen? Wir haben keine Wahl, die jedem gefällt, aber eine sinnvolle: Aufgaben `X` und `Y`, die **sich
gegenseitig erreichen**, sind in den alten Klausuren manchmal leichter und manchmal schwerer als die andere. Dann sagen wir: Sie sind **gleich
schwer**, und es ist egal, in welcher Reihenfolge sie stehen. Alle anderen Beziehungen müssen eingehalten werden.

Wer gleich schwer ist wie `X`, ist auch gleich schwer wie alle anderen, die gleich schwer sind wie `X`. Das bedeutet: Es entstehen **Gruppen**. In einer Gruppe erreicht
jeder jeden. Aufgaben ohne Konflikt bilden eine Gruppe für sich.

`finde_gruppen` geht die Aufgaben der Reihe nach durch. Ist `a` noch in keiner Gruppe, gehören alle `b` zu seiner Gruppe, die `a` erreichen und von `a` erreicht
werden (und `a` selbst).

In [ ]:
def finde_gruppen(aufgaben, schwerer):
    erreichbare = {}
    for a in aufgaben:
        erreichbare[a] = erreichbar(schwerer, a)
    gruppen = []
    eingeteilt = set()
    for a in aufgaben:
        if a not in eingeteilt:
            gruppe = [a]
            for b in aufgaben:
                if b != a and b in erreichbare[a] and a in erreichbare[b]:
                    gruppe.append(b)
            for b in gruppe:
                eingeteilt.add(b)
            gruppen.append(gruppe)
    return gruppen


gruppen = finde_gruppen(aufgaben, schwerer)
print(gruppen)
for gruppe in gruppen:
    if len(gruppe) > 1:
        print('Konflikt, gleich schwer:', gruppe)

[['A'], ['B'], ['C'], ['D'], ['E'], ['F', 'G']]
Konflikt, gleich schwer: ['F', 'G']


Die Aufgaben `F` und `G` bilden eine Gruppe. Alle anderen Aufgaben stehen allein.

## Schritt 6: Die Gruppen anordnen

Zwischen den Gruppen gibt es keine Kreise mehr (sonst gehörten die Aufgaben zur selben Gruppe). Wir bauen einen neuen Graphen: Die Knoten sind die Gruppen (ihre
Nummern in der Liste), und es gibt einen Pfeil von der Gruppe von `X` zur Gruppe von `Y`, wenn `X` direkt leichter als `Y` ist und beide in verschiedenen Gruppen
liegen. Den sortieren wir wie in Schritt 3 mit Kahns Algorithmus.

Aus den sortierten Gruppen entsteht die Anordnung: Wir gehen die Gruppen der Reihe nach durch, nehmen die gewünschten Aufgaben jeder Gruppe (in alphabetischer
Reihenfolge) und lassen alle anderen weg.

In [ ]:
def ordne_an(aufgaben, klausuren, gewaehlt):
    """Ergebnis: die Gruppen in der Reihenfolge der Anordnung (jede Gruppe nur mit den gewünschten Aufgaben, leere Gruppen fehlen)."""
    schwerer = baue_graph(aufgaben, klausuren)
    gruppen = finde_gruppen(aufgaben, schwerer)
    gruppe_von = {}
    for nr in range(len(gruppen)):
        for a in gruppen[nr]:
            gruppe_von[a] = nr
    gruppen_schwerer = {}
    for nr in range(len(gruppen)):
        gruppen_schwerer[nr] = set()
    for a in aufgaben:
        for b in schwerer[a]:
            if gruppe_von[a] != gruppe_von[b]:
                gruppen_schwerer[gruppe_von[a]].add(gruppe_von[b])
    reihenfolge = sortiere(list(range(len(gruppen))), gruppen_schwerer)
    ergebnis = []
    for nr in reihenfolge:
        gewuenschte = []
        for a in sorted(gruppen[nr]):
            if a in gewaehlt:
                gewuenschte.append(a)
        if len(gewuenschte) > 0:
            ergebnis.append(gewuenschte)
    return ergebnis


def anordnung_text(gruppen):
    teile = []
    for gruppe in gruppen:
        if len(gruppe) == 1:
            teile.append(gruppe[0])
        else:
            teile.append('[' + ' '.join(gruppe) + ']')
    return ' '.join(teile)


ergebnis = ordne_an(aufgaben, klausuren, gewaehlt)
print('gewünscht:', ' '.join(gewaehlt))
print('Anordnung:', anordnung_text(ergebnis))

gewünscht: B C D E F
Anordnung: B E D F C


Das ist die Anordnung aus der Aufgabe: **B E D F C**. In eckigen Klammern stehen Aufgaben, die gleich schwer sind. Hier gibt es keine, denn von `F` und `G` ist nur
`F` gewünscht. Die Sortierung wird zuerst für **alle** Aufgaben gemacht. Dass `E` vor `D` kommt, liegt an `A < E < D` (Klausur 3), und `B` kommt vor `E`, weil `B < A < E` über
`A` (Klausuren 1 und 3).

## Schritt 7: Die Anordnung nachprüfen

Zwei Dinge soll eine gute Anordnung erfüllen, und die prüfen wir unabhängig vom Verfahren, am Graphen:

1. Alle gewünschten Aufgaben kommen genau einmal vor.
2. Für alle Aufgaben `X`, `Y` aus der Anordnung gilt: Ist `X` **eindeutig** leichter als `Y` (`Y` ist von `X` erreichbar, aber `X` nicht von `Y`), steht
   `X` vor `Y`.

In [ ]:
def pruefe_anordnung(aufgaben, klausuren, gewaehlt, gruppen):
    schwerer = baue_graph(aufgaben, klausuren)
    liste = []
    for gruppe in gruppen:
        for a in gruppe:
            liste.append(a)
    if sorted(liste) != sorted(gewaehlt):
        return False                                       # eine Aufgabe fehlt oder kommt doppelt vor
    stelle = {}
    for i in range(len(liste)):
        stelle[liste[i]] = i
    for x in liste:
        schwerer_als_x = erreichbar(schwerer, x)
        for y in liste:
            if y in schwerer_als_x and x not in erreichbar(schwerer, y):
                if stelle[x] > stelle[y]:
                    return False                           # x ist eindeutig leichter als y, steht aber dahinter
    return True


print(pruefe_anordnung(aufgaben, klausuren, gewaehlt, ergebnis))
print(pruefe_anordnung(aufgaben, klausuren, gewaehlt, [['B'], ['D'], ['E'], ['F'], ['C']]))       # E und D vertauscht: falsch

True
False


## Schritt 8: Alle Beispiele

Für jedes Beispiel geben wir die gewünschten Aufgaben, die Anordnung und die Gruppen mit Konflikten aus (alle Gruppen mit mindestens zwei Aufgaben, auch von nicht
gewünschten Aufgaben). Auch die Prüfung läuft für jedes Ergebnis.

In [ ]:
ergebnisse = {}
for nr in range(6):
    aufgaben, klausuren, gewaehlt = lies_aufgabe('beispieldaten/schwierigkeiten' + str(nr) + '.txt')
    ergebnis = ordne_an(aufgaben, klausuren, gewaehlt)
    ergebnisse[nr] = (aufgaben, klausuren, gewaehlt, ergebnis)
    gruppen = finde_gruppen(aufgaben, baue_graph(aufgaben, klausuren))
    konflikte = []
    for gruppe in gruppen:
        if len(gruppe) > 1:
            konflikte.append(''.join(sorted(gruppe)))
    print('schwierigkeiten' + str(nr) + '.txt:')
    print('  gewünscht:', ' '.join(gewaehlt))
    print('  Anordnung:', anordnung_text(ergebnis))
    print('  Konflikte (gleich schwer):', konflikte)
    print('  Prüfung:', pruefe_anordnung(aufgaben, klausuren, gewaehlt, ergebnis))

schwierigkeiten0.txt:
  gewünscht: B C D E F
  Anordnung: B E D F C
  Konflikte (gleich schwer): ['FG']
  Prüfung: True
schwierigkeiten1.txt:
  gewünscht: A C D F G
  Anordnung: A C G F D
  Konflikte (gleich schwer): []
  Prüfung: True
schwierigkeiten2.txt:
  gewünscht: A B D E F G
  Anordnung: [A B D E] [F G]
  Konflikte (gleich schwer): ['ABCDE', 'FGH']
  Prüfung: True
schwierigkeiten3.txt:
  gewünscht: A B C D E F G H I J K L M N
  Anordnung: [A B C D] [H I L] [M N] E J F K G
  Konflikte (gleich schwer): ['ABCD', 'HIL', 'MN']
  Prüfung: True
schwierigkeiten4.txt:
  gewünscht: B W I N F
  Anordnung: B I F N W
  Konflikte (gleich schwer): ['DE', 'FPQR', 'GHIJS', 'KMNO', 'TVWXYZ']
  Prüfung: True
schwierigkeiten5.txt:
  gewünscht: A B C D E F G H I J K L M N O P Q R S T U V W X Y Z
  Anordnung: H R Z [C S] Q E K N [M O] J L F V [A D G P T U X] B W I Y
  Konflikte (gleich schwer): ['ADGPTUX', 'CS', 'MO']
  Prüfung: True


**Ergebnis:** Die Anordnungen stimmen mit den offiziellen Lösungshinweisen überein, soweit sie dort angegeben sind:

| Beispiel | Anordnung | Hinweise |
|---|---|---|
| 0 | `B E D F C` | genau diese Anordnung |
| 1 | `A C G F D` | `A [CG] F D`: `C` und `G` in beliebiger Reihenfolge |
| 2 | `[A B D E] [F G]` | `[ABDE][FG]` |
| 3 | `[A B C D]`, `[H I L]`, `[M N]`, dann `E J F K G` | beliebiges Verschachteln von `[ABCD] E F G`, `M N` und `[HIL] J K` |
| 4 | `B I F N W` | genau diese Anordnung |
| 5 | `[C S]`, `[M O]` und `[A D G P T U X]` als Gruppen | Verschachtelung von `H [SC] E N [OM] J L F V`, `R`, `Z Q K` und `[ADGPTUX] B W I Y` |

Wo mehrere Anordnungen möglich sind (`A [CG] F D`, die Reihenfolge innerhalb der Klammern, die Verschachtelung unabhängiger Teile), gibt das Programm eine davon aus. Die
Aufgabe verlangt nur eine gute Anordnung.

## Für Neugierige

**Warum nicht alle Anordnungen durchprobieren?** Bei `k` Aufgaben gibt es `k!` Anordnungen. Schon bei den 14 Aufgaben von `schwierigkeiten3.txt` sind das
über 87 Milliarden, und bei den 26 Aufgaben von `schwierigkeiten5.txt` eine Zahl mit 27 Stellen. Das Durchprobieren wäre hoffnungslos. Die Sortierung nutzt dagegen die
Beziehungen selbst, und ihre Rechenzeit ist ungefähr proportional zur Zahl der Aufgaben und Beziehungen.

**Schneller:** Unsere Gruppensuche prüft für jede Aufgabe die Erreichbarkeit mit einer eigenen Tiefensuche. Bei `m` Aufgaben ist das etwa `m` Tiefensuchen. Die Algorithmen
von **Tarjan** und **Kosaraju** finden alle starken Zusammenhangskomponenten in **einem** Durchlauf. Das lohnt sich erst bei sehr vielen Aufgaben.

**Andere Arten, mit Konflikten umzugehen:** Wir haben Aufgaben im Konflikt als gleich schwer behandelt. Anders wäre zum Beispiel, eine Anordnung zu suchen, die
**möglichst wenige** Aussagen der alten Klausuren verletzt, oder die Mehrheit zählen zu lassen, wenn zwei Klausuren sich widersprechen. Das ist deutlich schwieriger: Eine
Anordnung mit den wenigsten verletzten Beziehungen zu finden gilt als **NP-schwer**, die schnelle Lösung gibt es also vermutlich nicht.

**Warum alle Aufgaben sortieren?** Zwischen zwei gewünschten Aufgaben kann der einzige Beleg eine nicht gewünschte Aufgabe sein (wie `E` zwischen `A` und `G` in
`schwierigkeiten1.txt`). Darum bauen wir den Graphen immer über **alle** Aufgaben und lassen erst am Ende die nicht gewünschten weg.

**MiniZinc:** Hier haben wir keinen Abschnitt. Eine topologische Sortierung ist ein einfacher Algorithmus, für den kein Solver nötig ist.